# Custom MT Pair Results Notebook

Run inside `custom-mt-pairs/`. Loads 2-task custom MT result summaries with task ID.


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

ROOT = Path(".")
print("Working directory:", ROOT.resolve())


## 1. Load results


In [ ]:

summary_files = [p for p in ROOT.rglob("*_summary.csv") if "skipped" not in p.name.lower() and "checkpoint" not in p.name.lower()]
dfs = []
for p in summary_files:
    try:
        df = pd.read_csv(p)
        if {"config", "task_name", "success_rate"}.issubset(df.columns):
            exp = p.name.replace("_summary.csv", "")
            df["experiment"] = exp
            df["source_file"] = str(p)
            dfs.append(df)
    except Exception as e:
        print("Could not read", p, e)

data = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
print("Summary files:")
for p in summary_files:
    print(" -", p)
print("Loaded rows:", len(data))
display(data.head())


## 2. Tables and plots


In [ ]:

if data.empty:
    print("No compatible summary CSVs found.")
else:
    display(data.sort_values(["experiment", "config", "task_name"]))

    for exp in sorted(data["experiment"].unique()):
        d = data[data["experiment"] == exp]
        pivot = d.pivot_table(index="config", columns="task_name", values="success_rate")
        display(pivot)

        configs = list(pivot.index)
        tasks = list(pivot.columns)
        x = np.arange(len(configs))
        width = 0.8 / max(len(tasks), 1)

        fig, ax = plt.subplots(figsize=(10, 5))
        for i, task in enumerate(tasks):
            ax.bar(x + (i - (len(tasks)-1)/2) * width, pivot[task], width, label=task)
        ax.set_title(f"{exp}: Success Rate by Config/Task")
        ax.set_xlabel("Config")
        ax.set_ylabel("Success rate")
        ax.set_ylim(0, 1.05)
        ax.set_xticks(x)
        ax.set_xticklabels(configs)
        ax.grid(True, axis="y", alpha=0.3)
        ax.legend()
        plt.tight_layout()
        plt.show()

    ranking = (
        data.groupby(["experiment", "config"])
        .agg(
            mean_success=("success_rate", "mean"),
            min_success=("success_rate", "min"),
            mean_return=("avg_return", "mean") if "avg_return" in data.columns else ("success_rate", "mean"),
            total_episodes=("episodes", "sum") if "episodes" in data.columns else ("success_rate", "count"),
        )
        .reset_index()
        .sort_values(["experiment", "mean_success", "min_success"], ascending=[True, False, False])
    )
    display(ranking)
    display(ranking.groupby("experiment").head(1))


## 3. Save notebook outputs


In [ ]:

out_dir = Path("notebook_outputs")
fig_dir = out_dir / "figures"
out_dir.mkdir(exist_ok=True)
fig_dir.mkdir(exist_ok=True)

if not data.empty:
    data.to_csv(out_dir / "loaded_summary_results.csv", index=False)

    for exp in sorted(data["experiment"].unique()):
        d = data[data["experiment"] == exp]
        pivot = d.pivot_table(index="config", columns="task_name", values="success_rate")
        configs = list(pivot.index)
        tasks = list(pivot.columns)
        x = np.arange(len(configs))
        width = 0.8 / max(len(tasks), 1)

        fig, ax = plt.subplots(figsize=(10, 5))
        for i, task in enumerate(tasks):
            ax.bar(x + (i - (len(tasks)-1)/2) * width, pivot[task], width, label=task)
        ax.set_title(f"{exp}: Success Rate by Config/Task")
        ax.set_xlabel("Config")
        ax.set_ylabel("Success rate")
        ax.set_ylim(0, 1.05)
        ax.set_xticks(x)
        ax.set_xticklabels(configs)
        ax.grid(True, axis="y", alpha=0.3)
        ax.legend()
        plt.tight_layout()
        fig.savefig(fig_dir / f"{exp}_success_by_config_task.png", dpi=200, bbox_inches="tight")
        plt.close(fig)

print("Saved outputs to", out_dir)
